In [0]:
from config import load_config

cfg = load_config("config_databricks.yaml")

In [0]:
from pyspark.sql import functions as F
from config import load_config

cfg = load_config("config_databricks.yaml")

applications = (
    spark.table(cfg["fct_applications_table"])
    .filter(F.col("is_hired") == 1)
    .withColumn("hired_date", F.to_date(F.col("hired_date")))
    .withColumn("apply_date", F.to_date(F.col("apply_date")))
    .withColumn("time_to_hire_days", F.datediff(F.col("hired_date"), F.col("apply_date")))
    .filter(F.col("time_to_hire_days") >= 0)
)

job = spark.table(cfg["dim_job_table"])

final_df = (
    applications.alias("a")
    .join(job.alias("j"), F.col("a.job_id") == F.col("j.job_id"), "inner")
    .select(
        F.col("a.job_id").alias("job_id"),
        F.col("j.department").alias("department"),
        F.col("a.application_id").alias("application_id"),
        F.col("a.apply_date").alias("apply_date"),
        F.col("a.hired_date").alias("hired_date"),
        F.col("a.time_to_hire_days").alias("time_to_hire_days"),
    )
)

agg_job = final_df.groupBy("job_id").agg(F.avg("time_to_hire_days").alias("avg_time_to_hire_days"))
agg_dept = final_df.groupBy("department").agg(F.avg("time_to_hire_days").alias("avg_time_to_hire_days"))

# Write table
final_df.write.format("delta").mode("overwrite").saveAsTable(cfg["time_to_hire_detail_table"])
agg_job.write.format("delta").mode("overwrite").saveAsTable(cfg["job_time_to_hire_table"])
agg_dept.write.format("delta").mode("overwrite").saveAsTable(cfg["department_time_to_hire_table"])

In [0]:
spark.sql('SELECT COUNT(*) FROM fct_time_to_hire_detail;').show()
spark.sql('SELECT * FROM fct_time_to_hire_detail LIMIT 10;').show()
spark.sql('SELECT * FROM job_time_to_hire LIMIT 10;').show()
spark.sql('SELECT * FROM department_time_to_hire LIMIT 10;').show()